In [8]:
import os
import torch
import torch_pruning as tp
from ultralytics import YOLO
import mlflow
import torch.nn as nn
from typing import cast, Any
from ultralytics.nn.modules import C2f, Conv, Bottleneck, Detect

In [2]:
DAGSHUB_TOKEN = "4893e77fec7d32ca9090e4d472ec706354aedc07"
MY_USERNAME = "TQanh23"             # Your DagsHub username
ORIGINAL_OWNER = "danghoanghai042" # The username who owns the base model
REPO_NAME = "ppe_v2"

os.environ["MLFLOW_TRACKING_USERNAME"] = MY_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN

In [3]:
original_repo_uri = f"https://dagshub.com/{ORIGINAL_OWNER}/{REPO_NAME}.mlflow"
mlflow.set_tracking_uri(original_repo_uri)

target_run_id = "6830f4c921a04b73ab687d33e5267f59" # Run ID from the original owner's repo
if target_run_id == "YOUR_RUN_ID_HERE":
    target_run_id = input("Enter the Run ID from the original repo: ").strip()

print(f"Downloading model from {ORIGINAL_OWNER}'s repository...")

try:
    base_model_path = mlflow.artifacts.download_artifacts(
        run_id=target_run_id, 
        artifact_path="weights/best.pt", 
        dst_path="./downloads"
    )
    print(f"✅ Downloaded base model from original repo to: {base_model_path}")
except Exception as e:
    raise ValueError(f"Could not download model from {original_repo_uri}. Check permissions or Run ID.") from e

✅ Downloaded base model from original repo to: d:\ppe_cv\ppe_v2\downloads\best.pt


In [11]:
class C2f_v2(nn.Module):
    """Equivalent to YOLOv8's C2f, but avoids torch.chunk() so it can be pruned."""
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__()
        self.c = int(c2 * e)
        self.cv0 = Conv(c1, self.c, 1, 1)
        self.cv1 = Conv(c1, self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1)
        self.m = nn.ModuleList(Bottleneck(self.c, self.c, shortcut, g, k=((3, 3), (3, 3)), e=1.0) for _ in range(n))

    def forward(self, x):
        y = [self.cv0(x), self.cv1(x)]
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))

In [12]:
def replace_c2f_with_c2f_v2(module):
    """Recursively replaces all C2f layers with C2f_v2 and transfers weights + metadata."""
    for name, child_module in module.named_children():
        if isinstance(child_module, C2f):
            # Map parameters from the old C2f to the new C2f_v2
            c2f_v2 = C2f_v2(
                child_module.cv1.conv.in_channels,
                child_module.cv2.conv.out_channels,
                n=len(child_module.m),
                shortcut=child_module.m[0].add if len(child_module.m) > 0 else False,
                g=child_module.m[0].cv2.conv.groups if len(child_module.m) > 0 else 1,
                e=child_module.c / child_module.cv2.conv.out_channels
            )
            
            # Transfer the split weights for cv0 and cv1
            c2f_v2.cv0.conv.weight.data = child_module.cv1.conv.weight.data[:c2f_v2.c]
            if child_module.cv1.conv.bias is not None:
                c2f_v2.cv0.conv.bias.data = child_module.cv1.conv.bias.data[:c2f_v2.c]
                
            c2f_v2.cv1.conv.weight.data = child_module.cv1.conv.weight.data[c2f_v2.c:]
            if child_module.cv1.conv.bias is not None:
                c2f_v2.cv1.conv.bias.data = child_module.cv1.conv.bias.data[c2f_v2.c:]
                
            c2f_v2.cv2.load_state_dict(child_module.cv2.state_dict())
            c2f_v2.m.load_state_dict(child_module.m.state_dict())

            # ========================================================
            # FIX: Copy Ultralytics metadata attributes (f, i, etc.)
            # ========================================================
            for attr in ['f', 'i', 'type', 'np']:
                if hasattr(child_module, attr):
                    setattr(c2f_v2, attr, getattr(child_module, attr))
            
            setattr(module, name, c2f_v2)
        else:
            replace_c2f_with_c2f_v2(child_module)

In [13]:
print("\nLoading model and executing structured pruning...")
model = YOLO(base_model_path)
pytorch_model = cast(nn.Module, model.model)
pytorch_model.eval()

# APPLY THE FIX
print("Converting C2f modules to Prunable C2f_v2...")
replace_c2f_with_c2f_v2(pytorch_model)

example_inputs = torch.randn(1, 3, 640, 640)
base_metrics = tp.utils.count_ops_and_params(pytorch_model, example_inputs)
base_macs, base_params = base_metrics[0], base_metrics[1]

# Protect the Detection head
ignored_layers = []
for m in pytorch_model.modules():
    if isinstance(m, Detect):
        ignored_layers.append(m)

target_ratio = 0.20
pruner = tp.pruner.MagnitudePruner(
    pytorch_model,
    example_inputs=example_inputs,
    importance=tp.importance.MagnitudeImportance(p=1),
    iterative_steps=1,                # Execute in one pass
    pruning_ratio=target_ratio,       # Some versions require `ch_sparsity=target_ratio` instead
    ignored_layers=ignored_layers,
)

# Call step() normally, no interactive loop needed
pruner.step()

pruned_metrics = tp.utils.count_ops_and_params(pytorch_model, example_inputs)
pruned_macs, pruned_params = pruned_metrics[0], pruned_metrics[1]

print(f"Base MACs: {base_macs}, Base Params: {base_params}")
print(f"Pruned MACs: {pruned_macs}, Pruned Params: {pruned_params}")
print(f"Reduction: {100 * (1 - pruned_macs / base_macs):.2f}% MACs")


Loading model and executing structured pruning...
Converting C2f modules to Prunable C2f_v2...
Base MACs: 39452427600.0, Base Params: 25861531
Pruned MACs: 27408608800.0, Pruned Params: 17321064
Reduction: 30.53% MACs


In [14]:
my_fork_uri = f"https://dagshub.com/{MY_USERNAME}/{REPO_NAME}.mlflow"
mlflow.set_tracking_uri(my_fork_uri)
mlflow.set_experiment("YOLOv8_Pruning_Phase1")

with mlflow.start_run(run_name=f"Pruned_from_{target_run_id[:5]}"):
    macs_reduction = 1.0 - (pruned_macs / base_macs)
    params_reduction = 1.0 - (pruned_params / base_params)
    
    mlflow.log_params({
        "source_owner": ORIGINAL_OWNER,
        "source_run_id": target_run_id,
        "pruning_ratio": target_ratio,
        "algorithm": "L1_Magnitude"
    })
    
    mlflow.log_metrics({
        "original_params": base_params,
        "pruned_params": pruned_params,
        "params_reduction_pct": params_reduction * 100,
        "macs_reduction_pct": macs_reduction * 100
    })
    
    pruned_model_filename = "yolov8_pruned_20_untrained.pt"
    torch.save(pytorch_model, pruned_model_filename)
    
    print(f"Logging new pruned model to YOUR fork ({MY_USERNAME})...")
    mlflow.log_artifact(pruned_model_filename, artifact_path="pruned_models")
    
    print(f"\n✅ Done! The pruned model is now saved in your fork's MLflow dashboard.")

Logging new pruned model to YOUR fork (TQanh23)...

✅ Done! The pruned model is now saved in your fork's MLflow dashboard.
🏃 View run Pruned_from_6830f at: https://dagshub.com/TQanh23/ppe_v2.mlflow/#/experiments/0/runs/7e4672bb0a13403ab32859b40e9e57c2
🧪 View experiment at: https://dagshub.com/TQanh23/ppe_v2.mlflow/#/experiments/0
